In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/drw-crypto-market-prediction/sample_submission.csv
/kaggle/input/drw-crypto-market-prediction/train.parquet
/kaggle/input/drw-crypto-market-prediction/test.parquet


In [2]:
!rm -rf /kaggle/working/crypto_market_prediction
!git clone https://github.com/CapstoneTeam23UMICH/crypto_market_prediction.git
!pip install -r /kaggle/working/crypto_market_prediction/requirements.txt

Cloning into 'crypto_market_prediction'...
remote: Enumerating objects: 2816, done.
remote: Counting objects: 100% (2685/2685), done.
remote: Compressing objects: 100% (1988/1988), done.
remote: Total 2816 (delta 494), reused 2628 (delta 470), pack-reused 131 (from 1)
Receiving objects: 100% (2816/2816), 28.37 MiB | 19.54 MiB/s, done.
Resolving deltas: 100% (545/545), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 33.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 31.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 36.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━

In [3]:
import sys
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import shap
from tqdm import tqdm
import warnings

import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
import gc

sys.path.append('/kaggle/working/crypto_market_prediction')

from src.github_push_file import push_parquet_to_github
from src.github_push_folder import push_folder_to_github

from src.get_refresh_metadata import (
    get_feature_drift_df,
    get_correlation_train_df,
    get_correlation_test_df,
    get_autocorrelation_train_df,
    get_adversarial_validation_df,
    get_mutual_information_train_df,
    get_correlation_stability_df,
    get_vif_train_df)

from src.common import (
    evaluate_regression, 
    evaluate_classification, 
    preprocess_classifier_sae)

from src.get_feature_set import get_feature_set
from src.get_cv_splits import get_folds_for_model

from src.model_registry import (
    model_registry_regression,
    model_registry_autoencoder)

from src.model_fit_predict import (
    fit_predict_tree, 
    fit_predict_mlp, 
    fit_predict_classifier_sae)

from src.run_cv import (
    to_tensor,
    run_cv,
    expand_grid,
    run_grid_search
)

df_drift = get_feature_drift_df()
df_corr_train = get_correlation_train_df()
df_autocorr = get_autocorrelation_train_df()
df_adv_val = get_adversarial_validation_df()
df_mutual_information = get_mutual_information_train_df()
df_corr_stability = get_correlation_stability_df()
df_vif = get_vif_train_df()

Loading existing df_feature_drift.parquet
Loading existing corr_long_train.parquet
Loading existing autocorr_long_train.parquet
Loading existing df_adversarial_validation.parquet
Loading existing df_mutual_information_train.parquet
Loading existing df_corr_stability.parquet
Loading existing df_vif.parquet


In [5]:
train_path = '/kaggle/input/drw-crypto-market-prediction/train.parquet'
test_path = '/kaggle/input/drw-crypto-market-prediction/test.parquet'

df_train = pd.read_parquet(train_path).astype('float32')
df_test = pd.read_parquet(test_path).astype('float32')

In [6]:
known_features = ['bid_qty', 'ask_qty', 'buy_qty', 'sell_qty', 'volume']
target = 'label'
anonymized_features = sorted(list(set(df_train.columns) - set(known_features) - set([target])), key=lambda x: int(x[1:]))
selected_features = get_feature_set(anonymized_features, corr_thresh = 0.99, vif_tresh = 100, mi_tresh = 0.02)

Loading existing df_feature_drift.parquet
Loading existing corr_long_train.parquet
Loading existing autocorr_long_train.parquet
Loading existing df_adversarial_validation.parquet
Loading existing df_mutual_information_train.parquet
Loading existing df_vif.parquet
Returning Features Set of Size: 236


invalid value encountered in less


## Running Model Grid Search & Push to Github

In [ ]:
for i in ['MLP','XGB','LBGM']:
    result = run_grid_search(
        'MLP',
        df_train,
        selected_features,
        target="label",
        mode="find_best_model",
        tracking_uri= "/kaggle/working/crypto_market_prediction/mlruns",
        experiment_name = 'regression_grid_search',
    )

push_folder_to_github(
    repo_path="/kaggle/working/crypto_market_prediction",
    folder_to_commit="mlruns",
    commit_msg = "Add experiments for regression grid search",
    new_branch = 'mlruns/add-experiment-for-regression-grid-search-2',
    github_token = github_token,
    github_user=github_user,
    github_email=github_email,
    repo_name="crypto_market_prediction",
    base_branch="main"
)

In [7]:
result = run_grid_search(
    'classifier_SAE',
    df_train,
    selected_features,
    target="label",
    mode="find_best_model",
    tracking_uri= "/kaggle/working/crypto_market_prediction/mlruns",
    experiment_name = 'autoencoder_grid_search',
)

push_folder_to_github(
    repo_path="/kaggle/working/crypto_market_prediction",
    folder_to_commit="/kaggle/working/crypto_market_prediction/mlruns",
    commit_msg = "Add experiments for autoencoder grid search 2",
    new_branch = 'mlruns/add-experiment-for-autoencoder-grid-search-8',
    github_token = github_token,
    github_user=github_user,
    github_email=github_email,
    repo_name="crypto_market_prediction",
    base_branch="main"
)

2025/08/12 14:10:30 INFO mlflow.tracking.fluent: Experiment with name 'autoencoder_grid_search' does not exist. Creating a new experiment.
Running search for classifier_SAE: 100%|██████████| 32/32 [28:03<00:00, 52.61s/it]


## Hyperparameter Sensitivity Analysis

In [8]:
import mlflow
import pandas as pd

mlflow.set_tracking_uri("/kaggle/working/crypto_market_prediction/mlruns")

experiment_name = "autoencoder_grid_search"

experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment is None:
    raise ValueError(f"No experiment named '{experiment_name}' found.")

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    output_format="pandas"
)
